# Introduction

Idea is to fit a *single GEV distribution* using annual maxima from **multiple years (pooled together)**, not separate GEVs for each year

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

Estimated Time
|Workers | Estimated Time | Speedup| 
|---| ---| ---| 
|1 core (sequential) | ~45-60 minutes | 1x baseline| 
|3 cores | ~15-20 minutes | ~3x faster| 
| 4 cores | ~12-15 minutes | ~4x faster|

# Import Libraries

In [ ]:
from typing import Dict, Optional
from glob import glob
import psutil
import os
import multiprocessing as mp
import time

from tqdm import tqdm
import pickle
from pathlib import Path

from pandas import DataFrame, concat
from numpy import ndarray, sum, log, ndarray, full_like, any, inf,exp
from scipy import stats
from scipy.optimize import minimize
import random

import matplotlib.pyplot as plt
import seaborn as sns

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from time import sleep


import func_preparation as dbf
import func_stationary as dbst

sns.set_style('whitegrid')

# Settings

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'

In [ ]:
hindcast_start = 1960
hindcast_end = 2019

In [ ]:
geolocator = Nominatim(user_agent="geo_lookup")

In [ ]:
# For code parallelization later in the process 
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"Logical cores (mp): {mp.cpu_count()}")

Physical cores: 4
Logical cores: 8
Logical cores (mp): 8


# Import data

In [ ]:
ls_files = [file for file in glob(path + '*.nc')]

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files[:3]):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

Importing data from model MIROC6 (1/8)...
Importing data from model MPI-ESM1-2-HR (2/8)...
Importing data from model HadGEM3-GC31-MM (3/8)...


# Data Preparation 

BiasCorrection and Selection of valid data

In [ ]:
en = 0 
for model_name, dic_data_model in dic_data_per_model.items():
    print(f'Processing model: {model_name} ({en+1}/{len(ls_files)})...')

    ds_model_corrected = dbf.bias_correction(dic_data_model['raw data'])
    dbf.verify_bias_correction(dic_data_model['raw data'], ds_model_corrected)
    print(f' - Bias correction successfully done and verified.')

    data_valid, sites_valid, sites_total, rate_invalid = dbf.select_valid_data(
        dic_data_model['raw data'], ds_model_corrected
    )
    print(
        f" - Site Selection done:\n"
        f"\tFrom {sites_total} sites, {sites_valid} are model valid – {rate_invalid:.2f}% invalid sites removed."
    )
    
    dic_data_per_model[model_name]['valid data'] = data_valid
    dic_data_per_model[model_name]['preparation info'] = sites_valid, sites_total, rate_invalid
    en += 1
    print("-------------------------------------------------------------------------------------")
    print("")

Processing model: MIROC6 (1/8)...
 - Validating bias correction:
	Asserting whether actual ([nan nan]) selection matches expected selection ([nan nan])... 
 - Bias correction successfully done and verified.
 - Site Selection done:
	From 11022 sites, 7054 are model valid – 36.00% invalid sites removed.
-------------------------------------------------------------------------------------

Processing model: MPI-ESM1-2-HR (2/8)...
 - Validating bias correction:
	Asserting whether actual ([1.02845407 1.98558529]) selection matches expected selection ([1.02845407 1.98558529])... 
 - Bias correction successfully done and verified.
 - Site Selection done:
	From 11022 sites, 5808 are model valid – 47.31% invalid sites removed.
-------------------------------------------------------------------------------------

Processing model: HadGEM3-GC31-MM (3/8)...


In [ ]:
sites_valid = [dic_data_per_model[model_label]['preparation info'][0] for model_label in dic_data_per_model.keys()]

print("Data Overview")
print(
    f"Data from {len(ls_files)} models is available for "
    f"{min(sites_valid)}-{max(sites_valid)} locations"
    f""
)

# Workflow stationary GEV

### Utils

In [ ]:
def locations_label_lookup_with_cache(
    locations: DataFrame, cache_file: str = "geocoding_cache.pkl", max_retries: int = 3, delay: float = 1.5,
    user_agent: str = "storm_surge_analysis"
    ):
    """
    Geocode locations with caching to resume after interruptions.
    
    If the script fails or is interrupted, you can restart and it will
    continue from where it left off.
    
    Parameters:
    -----------
    locations : DataFrame
        Must have 'lat' and 'lon' columns
    cache_file : str
        File to save progress (default: "geocoding_cache.pkl")
    max_retries : int
        Maximum retry attempts per location
    delay : float
        Delay between requests in seconds
    user_agent : str
        Custom user agent for the geocoder
        
    Returns:
    --------
    list : Location objects (or None for failures)
    """
    geolocator = Nominatim(user_agent=user_agent, timeout=10)
    
    cache_path = Path(cache_file)
    if cache_path.exists():
        with open(cache_file, 'rb') as f:
            cache = pickle.load(f)
        print(f"✓ Loaded cache with {len(cache)} existing results")
    else:
        cache = {}
    
    locations_label = [None] * len(locations)
    failed_indices = []
    
    to_process = [ix for ix in locations.index if ix not in cache]
    already_done = len(locations) - len(to_process)
    
    print(f"Geocoding status:")
    print(f"  Already cached: {already_done}/{len(locations)}")
    print(f"  To process: {len(to_process)}")
    print(f"  Estimated time: ~{len(to_process) * delay / 60:.1f} minutes")
    
    try:
        for ix in tqdm(locations.index, desc="\t\tGeocoding"):
            if ix in cache:
                locations_label[ix] = cache[ix]
                continue
            
            lat = locations.loc[ix, 'lat']
            lon = locations.loc[ix, 'lon']
            
            location = None
            
            for attempt in range(max_retries):
                try:
                    location = geolocator.reverse((lat, lon), exactly_one=True, timeout=10)
                    break  
                    
                except GeocoderTimedOut:
                    if attempt < max_retries - 1:
                        sleep(2 ** attempt) 
                        continue
                    else:
                        failed_indices.append(ix)
                        
                except GeocoderServiceError:
                    failed_indices.append(ix)
                    break
                    
                except Exception as e:
                    print(f"\n⚠️  Error at index {ix}: {e}")
                    failed_indices.append(ix)
                    break
            
            locations_label[ix] = location
            cache[ix] = location 
            
            if (ix + 1) % 50 == 0:
                with open(cache_file, 'wb') as f:
                    pickle.dump(cache, f)
            
            sleep(delay)
    
    except KeyboardInterrupt:
        print("\nInterrupted by user. Saving progress...")
        with open(cache_file, 'wb') as f:
            pickle.dump(cache, f)
        print(f"✓ Progress saved to {cache_file}")
        print(f"  Run again to continue from index {ix}")
        raise
    
    with open(cache_file, 'wb') as f:
        pickle.dump(cache, f)
    
    success_count = sum(1 for loc in locations_label if loc is not None)
    print(f"\n✓ Geocoding complete:")
    print(f"  Success: {success_count}/{len(locations)}")
    print(f"  Failed: {len(failed_indices)}")
    
    return locations_label


def locations_label_lookup_batched(
    locations: DataFrame, batch_size: int = 100, cache_file: str = "geocoding_cache.pkl", delay: float = 1.5
    ):
    """
    Process locations in batches with automatic saving between batches.
    
    Parameters:
    -----------
    locations : DataFrame
        Must have 'lat' and 'lon' columns
    batch_size : int
        Number of locations to process before saving (default: 100)
    cache_file : str
        File to save progress
    delay : float
        Delay between requests in seconds
        
    Returns:
    --------
    list : Location objects (or None for failures)
    """
    cache_path = Path(cache_file)
    if cache_path.exists():
        with open(cache_file, 'rb') as f:
            cache = pickle.load(f)
        print(f"\t\t✓ Loaded {len(cache)} cached results")
    else:
        cache = {}
    
    geolocator = Nominatim(user_agent="storm_surge_analysis", timeout=10)
    
    total_batches = (len(locations) + batch_size - 1) // batch_size
    
    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(locations))
        
        print(f"\t\tProcessing batch {batch_num + 1}/{total_batches} (locations {start_idx}-{end_idx})")

        for ix in tqdm(range(start_idx, end_idx), desc=f"\t\tBatch {batch_num + 1}"):
            if ix in cache:
                continue
            
            lat = locations.loc[ix, 'lat']
            lon = locations.loc[ix, 'lon']

            try:
                location = geolocator.reverse((lat, lon), exactly_one=True, timeout=10)
                cache[ix] = location
            except Exception as e:
                print(f"\n⚠️  Error at {ix}: {e}")
                cache[ix] = None
            
            sleep(delay)

        with open(cache_file, 'wb') as f:
            pickle.dump(cache, f)
        print(f"\t\t✓ Batch {batch_num + 1} saved to cache")
    
    locations_label = [cache.get(ix) for ix in locations.index]
    
    success_count = sum(1 for loc in locations_label if loc is not None)
    print(f"\t\t✓ All batches complete: {success_count}/{len(locations)} successful")
    
    return locations_label


def locations_label_lookup(locations: DataFrame):
    locations_label = []
    time_outs = []
    to = 0
    for ix in locations.index:
        lat = locations.loc[ix, 'lat']
        lon = locations.loc[ix, 'lon']
        
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True)
        except GeocoderTimedOut:
            print(f"Timeout at index {ix}, retrying...")
            time_outs = to
            to +=1
            sleep(1)
            location = geolocator.reverse((lat, lon), exactly_one=True)
        
        locations_label.append(location)
        sleep(1) 
    return locations_label

In [ ]:
def prepare_data(data: DataFrame, hindcast_start:int, hindcast_end: int) -> DataFrame:
        """Calculate target years and filter to hindcast period."""
        data['target_year'] = data['sim_year'] + data['lead']

        mask = (data['target_year'] >= hindcast_start) & \
                (data['target_year'] <= hindcast_end)
        data_hindcast = data[mask].copy()

        print(f"\nData Summary:")
        print(f"  Hindcast period: {hindcast_start}-{hindcast_end}")
        print(f"  Total observations: {len(data_hindcast):,}")
        print(f"  Models: {data_hindcast['model'].nunique()}")
        print(f"  Locations: {min(data_hindcast[['lon', 'lat']].nunique().values)}")
        
        return data_hindcast

In [ ]:
def extract_annual_maxima(data_hindcast: DataFrame, model: str, lon: float, lat: float) -> DataFrame:
    """
    Extract annual maxima for a specific model-location combination.
    
    For each target year, takes maximum across all sim_year+lead combos.
    """

    subset = data_hindcast[
        (data_hindcast['model'] == model) & 
        (data_hindcast['lon'] == lon) &
        (data_hindcast['lat'] == lat)
    ].copy()

    if len(subset) == 0:
        return DataFrame(columns=['year', 'annual_max'])
    
    annual_max = subset.groupby('target_year')['storm_surge'].max().reset_index()
    annual_max.columns = ['year', 'annual_max']
    
    return annual_max.sort_values('year')


def fit_stationary_gev(data: ndarray) -> Dict:
    """
    Fit stationary GEV using Maximum Likelihood Estimation.
    
    This assumes GEV parameters are constant over time.
    RECOMMENDED: Use when you have 60+ years × 2 members = 120 points
    
    Parameters:
    -----------
    data : np.ndarray
        Annual maxima values
        
    Returns:
    --------
    dict : GEV parameters and diagnostics
    """
    if len(data) < 10:
        print(f"Warning: Only {len(data)} observations. Need at least 10 for reliable GEV fit.")
        return None
    
    try:
        c, loc, scale = stats.genextreme.fit(data)
        shape = -c 
        
        ll = sum(stats.genextreme.logpdf(data, c, loc, scale))
        
        n_params = 3
        aic = 2 * n_params - 2 * ll
        bic = log(len(data)) * n_params - 2 * ll
        
        if abs(shape) < 0.05:
            dist_type, tail = "Gumbel (Type I)", "Exponential"
        elif shape > 0:
            dist_type, tail = "Fréchet (Type II)", "Heavy (polynomial)"
        else:
            dist_type, tail = "Weibull (Type III)", "Light (bounded)"

        return {
            'shape': shape,
            'location': loc,
            'scale': scale,
            'n_obs': len(data),
            'log_likelihood': ll,
            'aic': aic,
            'bic': bic,
            'dist_type': dist_type,
            'tail_behavior': tail
        }
    except Exception as e:
        print(f"Failed to conduct GEV fitting due to error: {e}")
        return None

In [ ]:
def fit_nonstationary_gev(
    years: ndarray, data: ndarray, trend_params: str = 'location'
    ) -> Dict:
    """
    Fit non-stationary GEV where parameters vary linearly with time.
    
    This allows detection of trends in extreme values.
    RECOMMENDED: Use to test if sea level rise affects extremes
    
    Parameters:
    -----------
    years : np.ndarray
        Years corresponding to each observation
    data : np.ndarray
        Annual maxima values
    trend_params : str
        Which parameters have trends: 'location', 'scale', or 'both'
        
    Returns:
    --------
    dict : Non-stationary GEV parameters and diagnostics
    """
    if len(data) < 20:
        print(f"Warning: Non-stationary GEV needs ≥20 obs. Have {len(data)}.")
        return None
    
    t = (years - years.mean()) / years.std()
    
    def neg_log_likelihood(params):
        """Negative log-likelihood for optimization."""
        if trend_params == 'location':
            mu0, mu1, sigma, xi = params
            mu_t = mu0 + mu1 * t
            sigma_t = full_like(t, sigma)
        elif trend_params == 'scale':
            mu, sigma0, sigma1, xi = params
            mu_t = full_like(t, mu)
            sigma_t = sigma0 + sigma1 * t
        elif trend_params == 'both':
            mu0, mu1, sigma0, sigma1, xi = params
            mu_t = mu0 + mu1 * t
            sigma_t = sigma0 + sigma1 * t
        else:
            raise ValueError("trend_params must be 'location', 'scale', or 'both'")
        
        
        if any(sigma_t <= 0):
            return inf
        
        z = (data - mu_t) / sigma_t
        
        if abs(xi) < 1e-10:  # Gumbel case
            ll = -sum(log(sigma_t)) - sum(z) - sum(exp(-z))
        else:
            term = 1 + xi * z
            if any(term <= 0):
                return inf
            ll = (-sum(log(sigma_t)) - 
                    (1 + 1/xi) * sum(log(term)) - 
                    sum(term**(-1/xi)))
        
        return -ll
    
    stationary = fit_stationary_gev(data)
    if stationary is None:
        return None
    
    try:
        if trend_params == 'location':
            x0 = [stationary['location'], 0.0, stationary['scale'], stationary['shape']]
            result = minimize(neg_log_likelihood, x0, method='Nelder-Mead')
            mu0, mu1, sigma, xi = result.x
            params_out = {
                'mu0': mu0, 'mu1': mu1, 'sigma': sigma, 'xi': xi,
                'trend_in': 'location'
            }
            n_params = 4
            
        elif trend_params == 'scale':
            x0 = [stationary['location'], stationary['scale'], 0.0, stationary['shape']]
            result = minimize(neg_log_likelihood, x0, method='Nelder-Mead')
            mu, sigma0, sigma1, xi = result.x
            params_out = {
                'mu': mu, 'sigma0': sigma0, 'sigma1': sigma1, 'xi': xi,
                'trend_in': 'scale'
            }
            n_params = 4
            
        else:  
            x0 = [stationary['location'], 0.0, stationary['scale'], 0.0, stationary['shape']]
            result = minimize(neg_log_likelihood, x0, method='Nelder-Mead')
            mu0, mu1, sigma0, sigma1, xi = result.x
            params_out = {
                'mu0': mu0, 'mu1': mu1, 'sigma0': sigma0, 'sigma1': sigma1, 'xi': xi,
                'trend_in': 'both'
            }
            n_params = 5
        
        ll = -result.fun
        aic = 2 * n_params - 2 * ll
        bic = log(len(data)) * n_params - 2 * ll
        
        params_out.update({
            'n_obs': len(data),
            'log_likelihood': ll,
            'aic': aic,
            'bic': bic,
            'years_mean': years.mean(),
            'years_std': years.std()
        })
        
        return params_out
        
    except Exception as e:
        print(f"Non-stationary GEV fitting error: {e}")
        return None


def calculate_return_levels(
    gev_params: Dict, return_periods: list = [10, 50, 100], year: Optional[float] = None
    ) -> Dict:
    """
    Calculate return levels from GEV parameters.
    
    Parameters:
    -----------
    gev_params : dict
        GEV parameters (stationary or non-stationary)
    return_periods : list
        Return periods in years
    year : float, optional
        For non-stationary: year to calculate return level
        
    Returns:
    --------
    dict : Return levels
    """
    if gev_params is None:
        return None
    
    if 'trend_in' in gev_params:
        if year is None:
            year = gev_params['years_mean']
        
        t = (year - gev_params['years_mean']) / gev_params['years_std']
        
        if gev_params['trend_in'] == 'location':
            mu = gev_params['mu0'] + gev_params['mu1'] * t
            sigma = gev_params['sigma']
            xi = gev_params['xi']
        elif gev_params['trend_in'] == 'scale':
            mu = gev_params['mu']
            sigma = gev_params['sigma0'] + gev_params['sigma1'] * t
            xi = gev_params['xi']
        else:  # both
            mu = gev_params['mu0'] + gev_params['mu1'] * t
            sigma = gev_params['sigma0'] + gev_params['sigma1'] * t
            xi = gev_params['xi']
    else:
        mu = gev_params['location']
        sigma = gev_params['scale']
        xi = gev_params['shape']
    
    return_levels = {}
    for T in return_periods:
        p = 1 - 1/T
        
        if abs(xi) < 1e-10:  # Gumbel
            z_p = mu - sigma * log(-log(p))
        else:
            z_p = mu + (sigma / xi) * ((-log(p))**(-xi) - 1)
        
        return_levels[f'{T}-year'] = z_p
    
    return return_levels


def compare_models(stationary: Dict, nonstationary: Dict) -> Dict:
    """
    Compare stationary vs non-stationary GEV using likelihood ratio test.
    
    Returns:
    --------
    dict : Test results and recommendation
    """
    if stationary is None or nonstationary is None:
        return None
    
    lr_statistic = 2 * (nonstationary['log_likelihood'] - stationary['log_likelihood'])
    
    if nonstationary['trend_in'] in ['location', 'scale']:
        df = 1  
    else:  
        df = 2  
    
    p_value = 1 - stats.chi2.cdf(lr_statistic, df)
    
    delta_aic = nonstationary['aic'] - stationary['aic']
    
    if p_value < 0.05:
        decision = "Non-stationary model is significantly better (p < 0.05)"
        recommendation = "Use non-stationary model - trend detected!"
    elif delta_aic < -2:
        decision = "Non-stationary preferred by AIC (ΔAIC < -2)"
        recommendation = "Use non-stationary model"
    else:
        decision = "No strong evidence for non-stationarity"
        recommendation = "Use stationary model (simpler)"
    
    return {
        'lr_statistic': lr_statistic,
        'df': df,
        'p_value': p_value,
        'delta_aic': delta_aic,
        'delta_bic': nonstationary['bic'] - stationary['bic'],
        'decision': decision,
        'recommendation': recommendation
    }


In [ ]:
def analyze_location(data_hindcast:DataFrame, model: str, lat: float, lon: float, location_info: str) -> Dict:
        """
        Complete analysis for one model-location combination.
        Fits both stationary and non-stationary GEV.
        """
        annual_max = extract_annual_maxima(data_hindcast, model=model, lon=lon, lat=lat)

        if len(annual_max) < 10:
                return None

        years = annual_max['year'].values
        data = annual_max['annual_max'].values

        print("\t\tconducting stationary GEV...")
        gev_stationary = fit_stationary_gev(data)
        print(f"\t\t\tstationary GEV done (success {gev_stationary != None}); continuing with non-stationary GEV...")
        gev_nonstat_loc = fit_nonstationary_gev(years, data, 'location')
        print(f"\t\t\tnon-stationary GEV done (success {gev_nonstat_loc != None}).")
        comparison = compare_models(gev_stationary, gev_nonstat_loc)

        rl_stationary = calculate_return_levels(gev_stationary, [10, 25, 50, 100, 200])
        rl_nonstat_start = calculate_return_levels(
                gev_nonstat_loc, [10, 50, 100], year=years.min()
        ) if gev_nonstat_loc else None

        rl_nonstat_end = calculate_return_levels(
                gev_nonstat_loc, [10, 50, 100], year=years.max()
        ) if gev_nonstat_loc else None

        return {
                'model': model,
                'location': (lon, lat),
                'location info': location_info,
                'annual_maxima': annual_max,
                'gev_stationary': gev_stationary,
                'gev_nonstationary': gev_nonstat_loc,
                'model_comparison': comparison,
                'return_levels_stationary': rl_stationary,
                'return_levels_1960': rl_nonstat_start,
                'return_levels_2019': rl_nonstat_end
        }


## Initial try 

### Batch data to 5 locations all years

Create a batch of 5 locations. If this workflow works, we can batch and paralize the rest.

In [ ]:
ls_models = list(dic_data_per_model.keys())
ls_models

In [ ]:
dic_model_valid = dict(
    map(
        lambda ex_model: (ex_model, dic_data_per_model[ex_model]['valid data']),
        ls_models
    )
)

In [ ]:
dic_locationsID_per_model = {}

for model_ in dic_model_valid.keys():
    locations_id = []
    while len(locations_id) < 5:
        locations_id.append(random.randint(0, dic_model_valid[model_].shape[-1]))    
    dic_locationsID_per_model[model_] = locations_id
    
dic_locationsID_per_model

In [ ]:
dic_batch_locations = {
    model_: dic_model_valid[model_][:, :, dic_locationsID_per_model[model_]] 
    for model_ in dic_model_valid.keys()
    }

dic_batch_locations.keys()

In [ ]:
dic_data = {}

for model_ in dic_batch_locations.keys():
    df = dic_batch_locations[model_].to_dataframe(name="value").reset_index()
    df['model'] = model_
    df.rename(columns={'value': 'storm_surge'}, inplace=True)
    dic_data[model_] = df
    
df = concat(dic_data).reset_index()

df.describe()

### Run Analysis

In [ ]:
models: Optional[list] = None
locations: Optional[list] = None
time_start = time.time()

print("\n" + "="*70)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER MODEL")
print("="*70)

analyzer = dbst.StormSurgeGEVAnalysis(df, hindcast_start=1960, hindcast_end=2019)

# ---------------------------------------------
df_prepared = prepare_data(data=df, hindcast_start=hindcast_start, hindcast_end=hindcast_end)

# ---------------------------------------------
if models is None:
    models = df_prepared['model'].unique()

locations_total = df_prepared[['lon', 'lat']].drop_duplicates().reset_index()

results = {}
count = 0

print(f"\nAnalyzing {len(models)} model(s) and {len(locations_total)} different locations...")

for en_m, model_ in enumerate(models):
    print(f"\nProcessing model {model_} ({en_m+1}/{len(models)})")

    print("\tLookup location info for batch...")
    locations = df_prepared[df_prepared.model == model_][['lon', 'lat']].drop_duplicates().reset_index()
    
    locations_label = locations_label_lookup_batched(locations) 
    total = len(models) * len(locations)

    results[model_] = {}
    for en in locations.index:
        location_info = locations_label[en]
        print(f"\n\tAnalyse location {location_info[0]} (ID {locations_id[en]}) ")
        
        count += 1
        if count % 100 == 0 or count == total:
            print(f"  Progress: {count}/{total} ({100*count/total:.1f}%)")

        result = analyze_location(df_prepared, model_, locations.loc[en].lat, locations.loc[en].lon, location_info[0])

        if result is None:
            print(f"\t\t→ Warning! No results found, skipping...")
        else:
            results[model_][(locations.loc[en].lat, locations.loc[en].lon)] = result
            print(f"\t\t→ Results produced successfully; storing to dictionary...")

print("\n" + "="*70)
time_end = time.time()
print(f"✓ ANALYSIS COMPLETED IN {(time_end - time_start):.2f}s!")
print("="*70)

### Show results for one location


In [ ]:
results.keys()

In [ ]:
result_display = None

example_model = input('select model ')

if example_model:
    print(f"selected model {example_model} for result display")
    
    result = results[example_model]

    location_in_model = [results[example_model][loc_]['location info'] for loc_ in list(result.keys())]
    print(f"locations available in model: \n{location_in_model}")
    location_choice = input('select location ')
    result_display = result[location_choice]
    
else:
    example_model = random.choice(list(random.choice(results.keys())))
    result = results[example_model]

    location_in_model = [results[example_model][loc_]['location info'] for loc_ in list(result.keys())]
    location_choice = random.choice(location_in_model)
    result_display = result[location_choice]

In [ ]:
if result_display:
    print("\n" + "="*70)
    print(f"RESULTS for {example_model} - {location_choice}")
    print("="*70)
    print(f"\nAnnual maxima: {len(result['annual_maxima'])} years")
    print(f"Observations per year: ~2 (from ensemble members)")
    print(f"Total data points for GEV: {result['gev_stationary']['n_obs']}")

    print("\nSTATIONARY GEV:")
    stat = result['gev_stationary']
    print(f"  μ (location) = {stat['location']:.3f}")
    print(f"  σ (scale) = {stat['scale']:.3f}")
    print(f"  ξ (shape) = {stat['shape']:.3f}")
    print(f"  Type: {stat['dist_type']}")


    if result['gev_nonstationary']:
        print("\nNON-STATIONARY GEV:")
        nonstat = result['gev_nonstationary']
        print(f"  μ(t) = {nonstat['mu0']:.3f} + {nonstat['mu1']:.4f}·t")
        print(f"  Trend = {nonstat['mu1'] * nonstat['years_std']:.4f} m/year")

    if result['model_comparison']:
        print("\nMODEL COMPARISON:")
        comp = result['model_comparison']
        print(f"  p-value: {comp['p_value']:.4f}")
        print(f"  Decision: {comp['decision']}")
        print(f"  → {comp['recommendation']}")


    print("\nVisualize results")
    analyzer.plot_analysis(example_model, example_location)

In [ ]:
def plot_analysis(results: dict, model: str, lat_lon_tuple: (float, float), save_path: str = None):
        """Create comprehensive visualization."""
        if model not in results or lat_lon_tuple not in results[model]:
            print(f"No results for {model}, {lat_lon_tuple}")
            return
        
        result = results[model][lat_lon_tuple]
        annual_max = result['annual_maxima']
        stat = result['gev_stationary']
        nonstat = result['gev_nonstationary']
        comp = result['model_comparison']
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle(f'GEV Analysis: {model} - {lat_lon_tuple}', fontsize=16, fontweight='bold')
        
        # Plot 1: Annual maxima with trends
        ax = axes[0, 0]
        ax.plot(annual_max['year'], annual_max['annual_max'], 
               'o', color='steelblue', markersize=6, alpha=0.6, label='Annual max')
        
        # Add return levels
        if result['return_levels_stationary']:
            for period in ['10-year', '50-year', '100-year']:
                level = result['return_levels_stationary'][period]
                ax.axhline(y=level, linestyle='--', alpha=0.5, 
                          label=f'{period} (stationary)')
        
        # Add non-stationary trend if significant
        if nonstat and comp and comp['p_value'] < 0.05:
            years_plot = np.linspace(annual_max['year'].min(), 
                                    annual_max['year'].max(), 100)
            t_plot = (years_plot - nonstat['years_mean']) / nonstat['years_std']
            mu_plot = nonstat['mu0'] + nonstat['mu1'] * t_plot
            ax.plot(years_plot, mu_plot, 'r-', linewidth=2.5, 
                   label='Non-stationary μ(t)', alpha=0.8)
        
        ax.set_xlabel('Year')
        ax.set_ylabel('Storm Surge (m)')
        ax.set_title(f'Annual Maximum Storm Surge\n(n={len(annual_max)} years)')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        
        # Plot 2: Model comparison
        ax = axes[0, 1]
        if stat and nonstat and comp:
            models_names = ['Stationary', 'Non-Stationary']
            aics = [stat['aic'], nonstat['aic']]
            colors = ['steelblue', 'darkred']
            
            bars = ax.bar(models_names, aics, color=colors, alpha=0.7, edgecolor='black')
            ax.set_ylabel('AIC (lower is better)')
            ax.set_title(f'Model Comparison\np = {comp["p_value"]:.4f}')
            
            # Add values on bars
            for bar, aic in zip(bars, aics):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{aic:.1f}', ha='center', va='bottom')
            
            # Add decision box
            decision_color = 'green' if comp['p_value'] < 0.05 else 'gray'
            ax.text(0.5, 0.95, comp['recommendation'], 
                   transform=ax.transAxes, fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor=decision_color, alpha=0.3),
                   horizontalalignment='center')
        
        # Plot 3: Return levels evolution
        ax = axes[1, 0]
        if result['return_levels_1960'] and result['return_levels_2019']:
            periods = ['10-year', '50-year', '100-year']
            levels_1960 = [result['return_levels_1960'][p] for p in periods]
            levels_2019 = [result['return_levels_2019'][p] for p in periods]
            
            x = np.arange(len(periods))
            width = 0.35
            
            ax.bar(x - width/2, levels_1960, width, label='1960', 
                  color='lightblue', edgecolor='black')
            ax.bar(x + width/2, levels_2019, width, label='2019', 
                  color='darkred', edgecolor='black', alpha=0.7)
            
            ax.set_xlabel('Return Period')
            ax.set_ylabel('Return Level (m)')
            ax.set_title('Return Levels: 1960 vs 2019')
            ax.set_xticks(x)
            ax.set_xticklabels(periods)
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')
        
        # Plot 4: GEV parameters summary
        ax = axes[1, 1]
        ax.axis('off')
        
        info_text = f"STATIONARY GEV:\n"
        if stat:
            info_text += f"  μ = {stat['location']:.3f}\n"
            info_text += f"  σ = {stat['scale']:.3f}\n"
            info_text += f"  ξ = {stat['shape']:.3f}\n"
            info_text += f"  Type: {stat['dist_type']}\n"
            info_text += f"  AIC = {stat['aic']:.1f}\n\n"
        
        if nonstat:
            info_text += f"NON-STATIONARY GEV:\n"
            info_text += f"  μ(t) = {nonstat['mu0']:.3f} + {nonstat['mu1']:.4f}·t\n"
            info_text += f"  σ = {nonstat['sigma']:.3f}\n"
            info_text += f"  ξ = {nonstat['xi']:.3f}\n"
            info_text += f"  AIC = {nonstat['aic']:.1f}\n\n"
        
        if comp:
            info_text += f"SIGNIFICANCE TEST:\n"
            info_text += f"  p-value = {comp['p_value']:.4f}\n"
            info_text += f"  ΔAIC = {comp['delta_aic']:.1f}\n"
        
        ax.text(0.1, 0.9, info_text, transform=ax.transAxes, 
               fontsize=10, verticalalignment='top', 
               family='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
print("\nVisualize results")
analyzer.plot_analysis(example_model, example_location)